# Lesson 4: Flood Fill and Connected Components

Once we have a binary image (e.g. from thresholding, see Lesson 3), a natural next question is: *how many separate blobs are there, and where are they?* Two tools answer this:

- **Flood fill** grows a region from a single seed pixel, spreading to all connected neighbors that share a similar value.
- **Connected component labeling** finds *all* such regions in a binary image at once, labeling each with a unique ID.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Build a test image with several blobs

We draw a few disconnected shapes plus one pair of shapes that touch, so we can see how connectivity is decided.

In [ ]:
binary = np.zeros((150, 200), dtype=np.uint8)
cv2.circle(binary, (40, 40), 25, 255, -1)          # blob 1
cv2.rectangle(binary, (100, 20), (140, 60), 255, -1)  # blob 2
cv2.circle(binary, (60, 110), 20, 255, -1)          # blob 3a
cv2.circle(binary, (95, 110), 20, 255, -1)          # blob 3b, touches 3a
cv2.circle(binary, (170, 120), 12, 255, -1)         # blob 4, small

plt.imshow(binary, cmap='gray')
plt.title('Binary test image')
plt.axis('off')
plt.show()

## Flood fill from a seed point

`cv2.floodFill` starts at a seed pixel and spreads outward to all connected pixels within a tolerance of the seed value, painting them a new color. Here we seed inside the touching pair of circles: flood fill treats them as *one* region because they are physically connected, even though we drew them as two separate `cv2.circle` calls.

In [ ]:
# floodFill needs a mask 2 pixels larger than the image, and modifies the image in place
filled = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
mask = np.zeros((binary.shape[0] + 2, binary.shape[1] + 2), dtype=np.uint8)
seed = (60, 110)  # (x, y) inside the touching pair

cv2.floodFill(filled, mask, seed, (0, 140, 255))

plt.imshow(cv2.cvtColor(filled, cv2.COLOR_BGR2RGB))
plt.scatter(*seed, c='red', s=30, marker='x')
plt.title('Flood fill from one seed (red x)')
plt.axis('off')
plt.show()

## Connected components: label every blob at once

Instead of picking seeds by hand, `cv2.connectedComponentsWithStats` scans the whole image and assigns every blob its own integer label, plus handy stats (bounding box, area, centroid) for free.

In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

print(f'Found {num_labels - 1} blobs (plus the background as label 0)')
print()
print(f'{"label":>5} {"area":>6} {"centroid":>16}')
for label in range(1, num_labels):
    area = stats[label, cv2.CC_STAT_AREA]
    cx, cy = centroids[label]
    print(f'{label:>5} {area:>6} ({cx:6.1f}, {cy:6.1f})')

Note that the touching pair of circles is reported as a *single* blob with one label, just as flood fill found it to be one connected region.

In [ ]:
# Give each label a distinct random color for visualization
rng = np.random.default_rng(1)
colors = rng.integers(50, 255, size=(num_labels, 3))
colors[0] = 0  # background stays black

colored = colors[labels].astype(np.uint8)

plt.imshow(colored)
for label in range(1, num_labels):
    cx, cy = centroids[label]
    plt.text(cx, cy, str(label), color='white', ha='center', va='center', fontsize=12, fontweight='bold')
plt.title('Connected components, colored and labeled')
plt.axis('off')
plt.show()

## Filtering blobs by size

A common use of connected components is to discard small, noise-like blobs and keep only significant ones — here, everything smaller than the small circle (blob 4).

In [ ]:
min_area = 500
keep_mask = np.zeros_like(binary)
for label in range(1, num_labels):
    if stats[label, cv2.CC_STAT_AREA] >= min_area:
        keep_mask[labels == label] = 255

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(binary, cmap='gray')
axes[0].set_title('All blobs')
axes[0].axis('off')
axes[1].imshow(keep_mask, cmap='gray')
axes[1].set_title(f'Blobs with area >= {min_area}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Exercise

1. Change `connectivity=8` to `connectivity=4` in `connectedComponentsWithStats`. Construct a binary image (e.g. a diagonal staircase of single pixels) where 4-connectivity and 8-connectivity give a *different* number of components.
2. Use `cv2.floodFill` with a nonzero `loDiff`/`upDiff` tolerance on a grayscale (not binary) image, and describe what changes.